# <center>Big Data &ndash; Exercise 7</center>
## <center>Fall 2026 &ndash; Week 7 &ndash; ETH Zurich</center>
## <center>Massive Parallel Processing - MapReduce practice (Optional)</center>

This exercise consists: 
* Java API example

## Important! ##

If you encounter an error like `Error response from daemon: Conflict. The container name "/resourcemanager" is already in use by container "344446f48f85ed6cb9ad46da4d9e27a3a2b93907008db80e9a191080e4c7008a". You have to remove (or rename) that container to be able to reuse that name.`, this is because, in the Week 3 exercise, we have a container with the same name. You can resolve the issue by removing the Week 3 containers with the command 'docker rm <container_id>' (or simply delete the containers using Docker Desktop).

This week's Docker setup already includes a JupyterLab instance. **You do not need to run Exam Magic Box to start JupyterLab.**

## 1. Setup a cluster

### Create an Hadoop cluster

Start the Hadoop cluster (in pseudo-distributed mode), similar to the HDFS exercise session, by running:

```bash
./run.sh
```

Wait for a couple minutes until the terminal no longer outputs startup logs.

You may see the following error:
```
resourcemanager  | .... Name node is in safe mode.
resourcemanager  | .... In safe mode extension. Safe mode will be turned off automatically in 24 seconds.
....
resourcemanager  | 2024-11-04 14:33:34,555 INFO resourcemanager.ResourceManager: SHUTDOWN_MSG: 
resourcemanager  | /************************************************************
resourcemanager  | SHUTDOWN_MSG: Shutting down ResourceManager at 7627975688dd/172.24.0.3
resourcemanager  | ************************************************************/
resourcemanager exited with code 255
```
In this case, wait at least 24 seconds. Then restart only the resource manager. If you run docker from a terminal, simply run `sudo docker-compose up` again, in parallel, and keep both the original and new terminal open.
(If you don't do this, and the ResourceManager is broken, MapReduce jobs may not work.)

## 2. Write a Word Count MapReduce job
We want to find which are the most frequently-used English words. To answer this question, we prepared a big text files (1.7GB) where we concatenated thousands of books of the [Gutenberg dataset](https://zenodo.org/record/3360392#.Y2gbl3aZNO-). In this exercise, a smaller text file `gutenberg_x0.1.txt` is used. 

### 2.1 Load the dataset

The dataset we provide consists of a concatenation of thousands of books (`gutenberg.txt`). However, we provide 2 versions, click the file below to download from Polybox. 

- [gutenberg_x0.1.txt](https://polybox.ethz.ch/index.php/s/9KLJZNRfLGJS7f3/download) - a smaller dataset of about 190MB
- [gutenberg.txt](https://polybox.ethz.ch/index.php/s/Lk9HkTdgpHYaBfw/download) - the original dataset, 1.7GB

**NOTE:** The process below will create up to 3 copies of the dataset: on your machine, in the NameNode, and in HDFS. Make sure you have enough disk space available. Otherwise, just use the small dataset.

Follow the steps below to set this dataset up in HDFS:

 - Transfer the dataset from your local computer to the NameNode container:
 
```bash
docker cp ./gutenberg_x0.1.txt namenode:gutenberg_x0.1.txt
docker cp ./gutenberg.txt namenode:gutenberg.txt
```

 - Log in to the NameNode container by launching a shell on it:
 
```bash
docker exec -it namenode /bin/bash
```

 - Load the dataset into the HDFS filesystem:
 
    With `ls -lh` you should see the 2 files mentioned above. These files are now in the "local" (remember, we are in containers) hard drive of your NameNode.

    Upload the files into HDFS where they can be consumed by MapReduce:

```bash
hdfs dfs -copyFromLocal *.txt /
``` 

### 2.2 Understand the MapReduce Java API

We wrote a template project that you can use to experiment with MapReduce named `mapreduce.zip`. Store **it on your local machine, not in the container**, and unzip the package.

**Note: Before you docker cp, make sure you remove the old mapreduce directory from the NameNode**

```bash
unzip mapreduce.zip
docker cp mapreduce namenode:/mapreduce
```

Now examine the content of the `./mapreduce/src` folder. You will see one Java class:
- *MapReduceWordCount*: a skeleton for a MapReduce job that loads data from file

Start looking at *MapReduceWordCount*. You can see that the *main* method is already provided. Our `WordCountMapper` and `WordCountReducer` are implemented as classes that extend Hadoop's `Mapper` and `Reducer`. For this exercise, you only need to consider (and override) the `map()` method for the mapper and the `reduce()` method for the reducer.

```java
public class Mapper<KEYIN, VALUEIN, KEYOUT, VALUEOUT> {
    protected void map(KEYIN key, VALUEIN value, Mapper<KEYIN, VALUEIN, KEYOUT, VALUEOUT>.Context context) {
        context.write(key, value);
    }
}
```

```java
public class Reducer<KEYIN, VALUEIN, KEYOUT, VALUEOUT> {
    protected void reduce(KEYIN key, Iterable<VALUEIN> values, Reducer<KEYIN, VALUEIN, KEYOUT, VALUEOUT>.Context context) {
        Iterator var4 = values.iterator();

        while(var4.hasNext()) {
            Object value = var4.next();
            context.write(key, value);
        }
    }
}
```

Consulting the [documentation](http://hadoop.apache.org/docs/stable/hadoop-mapreduce-client/hadoop-mapreduce-client-core/MapReduceTutorial.html) if necessary, answer the following questions:

1. What are possible types for `KEYIN, VALUEIN, KEYOUT and VALUEOUT`? Should `KEYOUT` and `VALUEOUT` for the Mapper be the same as `KEYIN` and `VALUEIN` for the Reducer?
1. What is the default behavior of a MapReduce job running the base Mapper and Reducer above?
1. What is the role of the object `Context`?

### 2.3 Write and run your MapReduce wordcount

Edit the provided skeleton and implement mapper and reducer to implement a word count. The goal is to know how many times each unique word appears in the dataset. You can consider each word as a sequence of characters separated by a whitespace, or implement a more sophisticated tokenizer if you wish.

**Hint**: to write the context you need to pass `Text` and `IntWritable` as types (normal `String` and `int` are not accepted).
You can use the `set` and `get` methods to convert from/to Java's standard types.

- Can you use your Reducer as Combiner? If so, enable it by uncommenting the appropriate line in the `main` method.

Once you are confident in your solution you can transfer it back to the container, compile it and run it from the **mapreduce/src** folder.

```bash
javac *.java -cp $(hadoop classpath)
jar cvf MapReduceWordCount.jar *.class 
```

Inside the folder, you will now find `MapReduceWordCount.jar`.
Run the map reduce job on the cluster using:

```bash
yarn jar MapReduceWordCount.jar MapReduceWordCount /gutenberg_x0.1.txt /tmp/results
```

To get the results after the job is done, simply copy the `results` directory from HDFS to the root of the NameNode:
```bash
cd /
hdfs dfs -copyToLocal /tmp/results
```

<br>The process is very similar to the one for HBase of last week. Answer the following questions:

1. Run the MapReduce job on the cluster with the default configuration and 4 DataNodes using only `gutenberg_x0.1.txt` for now. *(Note: if you want to run your job again, you first need to delete the previous result folder because Hadoop refuses to write in the same location)*: 
```bash
hdfs dfs -rm -r /tmp/results
```
2. How many map and reduce tasks were created with the default configuration?
3. Does it go faster with more reduce tasks? Uncomment and experiment with `job.setNumReduceTasks()`. What is the disadvantage of having multiple reducers? (Hint: check the format of your output)

### 2.4 Plot the results
*Note*: for this part we are working with only one reducer.
    
By default, output files have the form `part-A-XXXX` where `A` is *r* or *m* to denote Reducer or Mapper outputs, and `XXXX` is the id of the specific mapper or reducer task.

To plot the results, copy the file from the Docker container to your local machine, placing them next to this notebook in the `exercise07` (or `exercise07-aarch64`) directory, and then run the script below.

```bash
docker cp namenode:/results results
```

In [ ]:
import matplotlib.pyplot as plt
import operator
print ('Plotting...')
freq = {}

# Read input and sort by frequency. Keep only top 30.
# Read Reducer Output
with open('./results/part-r-00000', 'rb') as csvfile:
    for line in csvfile.readlines():
        word, count = line.decode('UTF-8').split('\t')
        freq[word] = int(count)
srt = sorted(freq.items(), key=operator.itemgetter(1), reverse=True)[:30]

# Generate plot
plt.figure(figsize=(16,6))
plt.bar(range(len(srt)), [x[1] for x in srt], align='center', color='#ba2121')
plt.xticks(range(len(srt)), [x[0] for x in srt])
plt.show()

If everything is correct, the 3 most frequent words should be **`the`**, **`of`** and **`and`**.

## 3. Performance comparison

- Test your MapReduce with `gutenberg_x0.1.txt` and `gutenberg.txt`. For each test, **write down the running time**:

```bash
time yarn jar MapReduceWordCount.jar MapReduceWordCount /gutenberg_x0.1.txt /tmp/gut01/
time yarn jar MapReduceWordCount.jar MapReduceWordCount /gutenberg.txt /tmp/gut1/
```

- For reference, it takes 34s for gutenberg_x0.1.txt and it takes 3m6s for gutenberg.txt.

- We prepared a simple [wordcount program](./wordcount.py) in Python in this folder. Download it on your laptop (or the cluster NameNode) and test how long it takes to process these two datasets. **Annotate the times for the next exercise.**
    
```bash
python wordcount.py < gutenberg_x0.1.txt
python wordcount.py < gutenberg.txt
```

- For reference, it takes 5.724s for gutenberg_x0.1.txt in my local workstation and it takes 42.984s for gutenberg.txt. 

### 3.1 Plot
Compare the performance of the MapReduce vs the single-thread implementation of the word count algorithm for two different input sizes. Fill the time in seconds in the code below to plot the results.

In [ ]:
size_input = [1.9*10e2, 1.7*10e3] # the input size in MB
time_mapreduce = [0, 0] # replace 0s with the time (in seconds) for the corresponding inputs
time_locallaptop = [0, 0] # replace 0s with the time (in seconds) for the corresponding inputs


%matplotlib inline
# Import plot library
import matplotlib.pyplot as plt
# Plot
plt.figure(figsize=(18,9))
plt.plot(size_input, time_mapreduce, '#f37626', label='MapReduce', linewidth=3.0)
plt.plot(size_input, time_locallaptop, '#00b300', label='Local laptop', linewidth=3.0, linestyle='dashed')

plt.xscale('log')
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.xlabel('Input size (MB)')
plt.ylabel('Time (seconds)')
plt.title('Wall-time comparison')
plt.show()

### 3.2 Discussion

We have run some more tests. Here we present the running time for 3 configurations on HDInsight, one workstation and one laptop. The figures below are indicative only, because the performance of every machine depends on several factors.

- **MapReduce v1**: no combiner with default configuration (1 reducer)
- **MapReduce v2**: no combiner with 8 reduce tasks
- **MapReduce v3**: using combiner with default configuration (1 reducer)
- **MapReduce v4**: using combiner with 8 reduce tasks
- **Workstation**: using a local workstation (server)
- **Laptop**: using a local laptop

Relate each of these categories to a line on the plot below.

<img src="https://polybox.ethz.ch/index.php/s/EK3qkWikLmcYfre/download" style="width:800px;">

1. Which line corresponds to which setting?
1. Which is faster, MapReduce on your cluster or a local wordcount implementation? Why?
2. Based on your experiment, what input size is the approximate break-even point for time performance?
3. Why MapReduce is not performing better than local computation for small inputs?
4. How can you optimize the MapReduce performance for this job?

In [ ]:
size_input = [1.2*1e2, 1.2*1e3, 1.2*1e4] # the input size in MB
time_mapreduce1 = [123, 445, 2160] # MapReduce v1
time_mapreduce2 = [100, 295, 970] # MapReduce v2
time_mapreduce3 = [110, 240, 413] # MapReduce v3
time_mapreduce4 = [110, 218, 385] # MapReduce v4
# 11.4, 127, 1332
#time_localstation = [11.4, 127, 1332] # our workstation (java)
time_localstation = [5.1, 47.7, 475.5] # our workstation
time_locallaptop = [14.9, 148.4, 1381.7] # our laptop


%matplotlib inline
# Import plot library
import matplotlib.pyplot as plt
# Plot
plt.figure(figsize=(18,9))
plt.plot(size_input, time_mapreduce1, '#7d0026', label='MapReduce v1', linewidth=3.0)
plt.plot(size_input, time_mapreduce2, '#f30000', label='MapReduce v2', linewidth=3.0)
plt.plot(size_input, time_mapreduce3, '#430064', label='MapReduce v3', linewidth=3.0)
plt.plot(size_input, time_mapreduce4, '#f37626', label='MapReduce v4', linewidth=3.0)

plt.plot(size_input, time_localstation, '#0055f3', label='Local workstation', linewidth=3.0, linestyle='dashed')
plt.plot(size_input, time_locallaptop, '#00b300', label='Local laptop', linewidth=3.0, linestyle='dashed')

plt.xscale('log')
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.xlabel('Input size (MB)')
plt.ylabel('Time (seconds)')
plt.title('Performance comparison')
plt.show()